In [ ]:
!rm -rf /content/movie_genre_classification
!git clone https://github.com/oluwoleadetifa/movie_genre_classification.git
%cd /content/movie_genre_classification

import sys
sys.path.append("/content/movie_genre_classification")

In [ ]:
!pip install transformers torch scikit-learn pandas numpy tqdm

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import os
import shutil

os.makedirs("/content/movie_genre_classification/data/processed", exist_ok=True)
os.makedirs("/content/movie_genre_classification/data/splits", exist_ok=True)

cwd = os.getcwd()
files_here = os.listdir(cwd)

def move_file(keyword, destination):
    matches = [f for f in files_here if keyword in f]
    if not matches:
        raise FileNotFoundError(f"Could not find file containing: {keyword}")
    shutil.move(os.path.join(cwd, matches[0]), destination)
    print(f"Moved {matches[0]} -> {destination}")

move_file("movies_with_posters", "/content/movie_genre_classification/data/processed/movies_with_posters.csv")
move_file("train", "/content/movie_genre_classification/data/splits/train.csv")
move_file("val", "/content/movie_genre_classification/data/splits/val.csv")
move_file("test", "/content/movie_genre_classification/data/splits/test.csv")

In [ ]:
import pandas as pd

train_df = pd.read_csv("/content/movie_genre_classification/data/splits/train.csv")
val_df = pd.read_csv("/content/movie_genre_classification/data/splits/val.csv")
test_df = pd.read_csv("/content/movie_genre_classification/data/splits/test.csv")

print(train_df.shape, val_df.shape, test_df.shape)
print(train_df.head())

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

TEXT_COLUMN = "description"
LABEL_COLUMN = "label"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert_model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
bert_model.eval()

In [ ]:
@torch.no_grad()
def encode_texts(texts, batch_size=16, max_length=256):
    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = list(texts[i:i + batch_size])

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}
        outputs = bert_model(**encoded)

        hidden = outputs.last_hidden_state
        mask = encoded["attention_mask"].unsqueeze(-1)

        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        embeddings.append(pooled.cpu().numpy())

    return np.vstack(embeddings)

In [ ]:
sample_embeddings = encode_texts(train_df[TEXT_COLUMN].tolist()[:8], batch_size=4)
print(sample_embeddings.shape)

In [ ]:
X_train_bert = encode_texts(train_df[TEXT_COLUMN].tolist(), batch_size=16)
X_val_bert = encode_texts(val_df[TEXT_COLUMN].tolist(), batch_size=16)
X_test_bert = encode_texts(test_df[TEXT_COLUMN].tolist(), batch_size=16)

y_train = train_df[LABEL_COLUMN].values
y_val = val_df[LABEL_COLUMN].values
y_test = test_df[LABEL_COLUMN].values

print(X_train_bert.shape, X_val_bert.shape, X_test_bert.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_bert, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = clf.predict(X_test_bert)

print("BERT Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
import json
import os

os.makedirs("/content/movie_genre_classification/outputs/metrics", exist_ok=True)

results_bert = {
    "test_accuracy": accuracy_score(y_test, y_pred),
    "classification_report": classification_report(y_test, y_pred, output_dict=True),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist()
}

with open("/content/movie_genre_classification/outputs/metrics/bert_results.json", "w") as f:
    json.dump(results_bert, f, indent=2)

print("BERT results saved.")

In [ ]:
import json
from google.colab import files

input_path = "/content/05_bert_text_model.ipynb"
output_path = "/content/05_bert_text_model_clean.ipynb"

with open(input_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

nb.get("metadata", {}).pop("widgets", None)

for cell in nb.get("cells", []):
    cell.get("metadata", {}).pop("widgets", None)
    cell.get("metadata", {}).pop("colab", None)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1)

files.download(output_path)